In [ ]:
import time

from boosteros.brain import Detection
from boosteros.robots.booster import BoosterRobot

# --- 控制参数微调 ---
MAX_VX = 0.3  # 最大前进速度 (m/s)
KP_YAW = 1.0  # 转向角速度增益 (P-Control)
MAX_VYAW = 0.8  # 最大旋转角速度 (rad/s)
CONFIDENCE_THRESHOLD = 0.4  # 检测置信度阈值

# 1. 初始化机器人与目标检测器
print("正在初始化机器人与检测模型...")
robot = BoosterRobot()
# 使用本地模式加载 Robocup (soccer) 模型
detector = Detection(model="soccer", backend="local")

In [ ]:
# 切换行走状态
if (cur_mode := robot.get_mode()) != "walk":
    print(f"Current mode is {cur_mode}, not walk")
    robot.set_mode("walk")
    time.sleep(1)

In [ ]:
# 初步检查图像获取是否正常
img = robot.get_image("default")
boxes = detector.detect(img.to_numpy())
detector.plot(img.to_numpy(), boxes)

In [ ]:
# 走向球
def go_to_ball():
     while True:
        # 获取图像并检测
        frame = robot.get_image()
        detections = detector.detect(frame.to_numpy(), confidence=CONFIDENCE_THRESHOLD)

        # 找球
        ball = max((d for d in detections if d.class_name == "Ball"), 
                   key=lambda d: d.confidence, default=None)

        if ball:
            # 有球：计算偏差并追踪
            error = (ball.bbox.center_x - frame.size()[0] / 2) / (frame.size()[0] / 2)
            vyaw = -error * MAX_VYAW * 0.5
            robot.set_velocity(vx=MAX_VX, vy=0.0, vyaw=vyaw)
        else:
            # 无球：固定方向旋转寻球
            robot.set_velocity(vx=0.0, vy=0.0, vyaw=MAX_VYAW * 0.5)

        time.sleep(0.05)


if __name__ == "__main__":
    go_to_ball()
